In [1]:
from glob import glob
import numpy as np
import torch
from src.sut import YoloSUT

from _analysis import load_jsons, get_class_stats, get_class_flip_stats, extract_class, align_mrm_lists, get_yolo_class_stats, load_img, get_origins_targets

%load_ext autoreload
%autoreload 2

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


### Load Data and paths

In [2]:
usr_path = "/home/weissl"

# Mimicry Data
mim_i_o_default, mim_i_t_default = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_class*", "/origin*.png", "/best*.png")
mim_i_o_eff, mim_i_t_eff = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_eff_class*", "/origin*.png", "/best*.png")
mim_i_o_vit, mim_i_t_vit = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_vit_class*", "/origin*.png", "/best*.png")
mim_c_o, mim_c_t = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*", "/origin*.png", "/best*.png")

# HyNeA Data
hyn_i_o_default, hyn_i_t_default = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*", "/origin*.png", "/taget*.png")
hyn_i_o_eff, hyn_i_t_eff = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_eff/*", "/origin*.png", "/taget*.png")
hyn_i_o_vit, hyn_i_t_vit = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_vit/*", "/origin*.png", "/taget*.png")
hyn_c_o, hyn_c_t = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*", "/origin*.png", "/taget*.png")
hyn_y_o, hyn_y_t = get_origins_targets(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*", "/origin*.png", "/taget*.png")

# GIFTBench Data
mrm_i_o_default, mrm_i_t_default = align_mrm_lists(
    *get_origins_targets(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_", "/*.png", "/**/*.npy")
)
mrm_i_o_eff, mrm_i_t_eff = align_mrm_lists(
    *get_origins_targets(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_eff", "/*.png", "/**/*.npy")
)
mrm_i_o_vit, mrm_i_t_vit = align_mrm_lists(
    *get_origins_targets(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_vit", "/*.png", "/**/*.npy")
)
mrm_c_o, mrm_c_t = align_mrm_lists(*get_origins_targets(f"{usr_path}/PycharmProjects/genai_tigs/sd_weights/celebahq_generatorlow", "/*.png", "/perturb-result/*.png"))
mrm_y_o, mrm_y_t = align_mrm_lists(*get_origins_targets(f"{usr_path}/PycharmProjects/genai_tigs/yolo_sd/test_", "/*.png", "/**/*.npy"))

# JSON data (SUT-level predictions)
mim_i = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_class*/*.json"))
mim_i_eff = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_eff_class*/*.json"))
mim_i_vit = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_vit_class*/*.json"))
mim_c = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*/*.json"))
hyn_i_default = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*/*.json"))
hyn_i_eff = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_eff/*/*.json"))
hyn_i_vit = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_vit/*/*.json"))
hyn_c = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*/*.json"))
hyn_y = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*/*.json"))
mrm_i = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_/*/*.json"))
mrm_i_eff = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_eff/*/*.json"))
mrm_i_vit = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_vit/*/*.json"))
mrm_c = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/sd_weights/celebahq_generatorlow/*.json"))

# Calculate Task Performance
### ImageNet

In [3]:
m_o_vals, m_t_vals = mim_i["w0_predictions"].values.tolist(), mim_i["best_0_y_hat"].values.tolist()
m_o_vals_eff, m_t_vals_eff = mim_i_eff["w0_predictions"].values.tolist(), mim_i_eff["best_0_y_hat"].values.tolist()
m_o_vals_vit, m_t_vals_vit = mim_i_vit["w0_predictions"].values.tolist(), mim_i_vit["best_0_y_hat"].values.tolist()
h_o_vals_default, h_t_vals_default = hyn_i_default["y_0"].values.tolist(), hyn_i_default["y_hat"].values.tolist()
h_o_vals_eff, h_t_vals_eff = hyn_i_eff["y_0"].values.tolist(), hyn_i_eff["y_hat"].values.tolist()
h_o_vals_vit, h_t_vals_vit = hyn_i_vit["y_0"].values.tolist(), hyn_i_vit["y_hat"].values.tolist()
mrm_o_vals, mrm_t_vals = mrm_i["initial_logits"].values.tolist(), mrm_i["final_logits"].values.tolist()
mrm_o_vals_eff, mrm_t_vals_eff = mrm_i_eff["initial_logits"].values.tolist(), mrm_i_eff["final_logits"].values.tolist()
mrm_o_vals_vit, mrm_t_vals_vit = mrm_i_vit["initial_logits"].values.tolist(), mrm_i_vit["final_logits"].values.tolist()

In [4]:
print("ImageNet Task Performance:")
m_misclass, m_escape = get_class_stats(m_o_vals, m_t_vals)
print(f"\tMimicry (WideResNet) Misclass Rate: {m_misclass:.3f}, Escape Ratio: {m_escape:.3f}")

m_misclass_eff, m_escape_eff = get_class_stats(m_o_vals_eff, m_t_vals_eff)
print(f"\tMimicry (EfficientNetV2) Misclass Rate: {m_misclass_eff:.3f}, Escape Ratio: {m_escape_eff:.3f}")

m_misclass_vit, m_escape_vit = get_class_stats(m_o_vals_vit, m_t_vals_vit)
print(f"\tMimicry (ViT) Misclass Rate: {m_misclass_vit:.3f}, Escape Ratio: {m_escape_vit:.3f}")

h_misclass, h_escape = get_class_stats(h_o_vals_default, h_t_vals_default)
print(f"\tHyNeA (WideResNet) Misclass Rate: {h_misclass:.3f}, Escape Ratio: {h_escape:.3f}")

h_misclass_eff, h_escape_eff = get_class_stats(h_o_vals_eff, h_t_vals_eff)
print(f"\tHyNeA (EfficientNetV2) Misclass Rate: {h_misclass_eff:.3f}, Escape Ratio: {h_escape_eff:.3f}")

h_misclass_vit, h_escape_vit = get_class_stats(h_o_vals_vit, h_t_vals_vit)
print(f"\tHyNeA (ViT) Misclass Rate: {h_misclass_vit:.3f}, Escape Ratio: {h_escape_vit:.3f}")

mrm_misclass, mrm_escape = get_class_stats(mrm_o_vals, mrm_t_vals)
print(f"\tGIFTBench (WideResNet) Misclass Rate: {mrm_misclass:.3f}, Escape Ratio: {mrm_escape:.3f}")

mrm_misclass_eff, mrm_escape_eff = get_class_stats(mrm_o_vals_eff, mrm_t_vals_eff)
print(f"\tGIFTBench (EfficientNetV2) Misclass Rate: {mrm_misclass_eff:.3f}, Escape Ratio: {mrm_escape_eff:.3f}")

mrm_misclass_vit, mrm_escape_vit = get_class_stats(mrm_o_vals_vit, mrm_t_vals_vit)
print(f"\tGIFTBench (ViT) Misclass Rate: {mrm_misclass_vit:.3f}, Escape Ratio: {mrm_escape_vit:.3f}")

ImageNet Task Performance:
	Mimicry (WideResNet) Misclass Rate: 0.820, Escape Ratio: 0.520
	Mimicry (EfficientNetV2) Misclass Rate: 0.950, Escape Ratio: 0.120
	Mimicry (ViT) Misclass Rate: 0.900, Escape Ratio: 0.250
	HyNeA (WideResNet) Misclass Rate: 1.000, Escape Ratio: 0.000
	HyNeA (EfficientNetV2) Misclass Rate: 1.000, Escape Ratio: 0.000
	HyNeA (ViT) Misclass Rate: 1.000, Escape Ratio: 0.000
	GIFTBench (WideResNet) Misclass Rate: 1.000, Escape Ratio: 0.842
	GIFTBench (EfficientNetV2) Misclass Rate: 1.000, Escape Ratio: 0.900
	GIFTBench (ViT) Misclass Rate: 1.000, Escape Ratio: 0.750


### CelebA

In [ ]:
hyn_c["cl"] = hyn_c["file"].apply(extract_class)
mim_c["cl"] = mim_c["file"].apply(extract_class)
mrm_c["cl"] = mrm_c["file"].apply(lambda x: x.split("_")[-1].split(".")[0])

m_o_vals, m_t_vals = mim_c["w0_predictions"].values.tolist(), mim_c["best_0_y_hat"].values.tolist()
h_o_vals, h_t_vals = hyn_c["y_0"].values.tolist(), hyn_c["y_hat"].values.tolist()
mrm_o_vals, mrm_t_vals = [e[0] for e in mrm_c["initial_logits"].values.tolist()], [e[0] for e in mrm_c["final_logits"].values.tolist()]

print("CelebA Task Performance:")
m_flip, m_sens = get_class_flip_stats(m_o_vals, m_t_vals, mim_c["cl"])
print(f"\tMimicry Flip Rate: {m_flip:.3f}, Flip Sensitivity: {m_sens:.3f}")

h_flip, h_sens = get_class_flip_stats(h_o_vals, h_t_vals, hyn_c["cl"])
print(f"\tHyNeA Flip Rate: {h_flip:.3f}, Flip Sensitivity: {h_sens:.3f}")

mrm_flip, mrm_sens = get_class_flip_stats(mrm_o_vals, mrm_t_vals, mrm_c["cl"])
print(f"\tGIFTBench Flip Rate: {mrm_flip:.3f}, Flip Sensitivity: {mrm_sens:.3f}")

### Yolo

In [3]:
sut = YoloSUT("yolov8n.pt", device=torch.device("cuda"), return_confidences=True, return_bboxes=False, objectness_exists=False)

In [ ]:
h_o_vals, h_t_vals = hyn_y["y_0"].values.tolist(), hyn_y["y_hat"].values.tolist()
g_o_vals = [sut.process_input(torch.tensor(load_img(p).transpose(2,0,1)).unsqueeze(0) / 255.).squeeze().tolist() for p in mrm_y_o]
g_t_vals = [sut.process_input(torch.tensor(load_img(p).transpose(2,0,1)).unsqueeze(0) / 255.).squeeze().tolist() for p in mrm_y_t]

y_targets = hyn_y["file"].apply(lambda x: x.split("_")[1]).values.tolist()
mrm_targets = [int(p.split("_")[-1].split(".")[0]) for p in mrm_y_o]
class_stats = get_yolo_class_stats(h_o_vals, h_t_vals, y_targets)
class_stats_mrm = get_yolo_class_stats(g_o_vals, g_t_vals, mrm_targets)

all_frac, all_frac_mrm = [], []
all_conf, all_conf_mrm = [], []

In [110]:
for cls, stats in class_stats.items():
    all_frac.extend(stats["frac"])
    all_conf.extend(stats["conf_decrease"])

all_frac = np.asarray(all_frac, dtype=float)
all_conf = np.asarray(all_conf, dtype=float)

print(f"\tHynea  Detections Removed: {all_frac.mean():.3f}pm{all_frac.std():.3f}, Confidence Decreased: {all_conf.mean():.3f} pm {all_conf.std():.3f}")

	Hynea  Detections Removed: 1.000pm0.000, Confidence Decreased: 0.947 pm 0.114


In [5]:
for cls, stats in class_stats_mrm.items():
    all_frac_mrm.extend(stats["frac"])
    all_conf_mrm.extend(stats["conf_decrease"])

all_frac_mrm = np.asarray(all_frac_mrm)
all_conf_mrm = np.asarray(all_conf_mrm)

print(f"\tGIFTbench Detections Removed: {all_frac_mrm.mean():.3f}pm{all_frac_mrm.std():.3f}, Confidence Decreased: {all_conf_mrm.mean():.3f} pm {all_conf_mrm.std():.3f}")

	GIFTbench Detections Removed: 0.719pm0.454, Confidence Decreased: 0.831 pm 0.252
